## dim_cliente - Transformações para silver

####Importando bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

### dim_cliente
####Consultando dados da tabela bronze

In [0]:
df_produto_credito = spark.table("dev_credit_risk.bronze.dim_produto_credito")
display(df_produto_credito)

#### Início das transformações
#### Retirando os espaços em branco das colunas de tipo string.

In [0]:
for field in df_produto_credito.schema.fields:
  if isinstance(field.dataType, StringType):
    df_produto_credito = df_produto_credito.withColumn(field.name, trim(col(field.name)))

df_produto_credito.display()

#### Normalizando os valores

In [0]:
df_produto_credito = (
    df_produto_credito
    .withColumn(
        "modalidadeCredito",
        F.when(F.upper(F.col("modalidadeCredito")).startswith("CRED"), "Crédito Pessoal")
         .when(F.upper(F.col("modalidadeCredito")).startswith("CRÉD"), "Crédito Pessoal")
         .otherwise(F.upper(F.col("modalidadeCredito")))
    )
)
df_produto_credito.display()

####Checando o domínio. Valores únicos após normalização

In [0]:
#checando os valores únicos.
display(df_produto_credito.select("modalidadeCredito").distinct())

###Padronizando os nomes das colunas

In [0]:
RENAME_MAP = {
    "idProduto": "id_produto",
    "modalidadeCredito": "modalidade_credito",
    "taxaJuros": "taxa_juros"
}

####Renomeando as colunas

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df_produto_credito = df_produto_credito.withColumnRenamed(old_name, new_name)


####Exibindo o dataframe pronto para ser inserido na tabela delta silver

In [0]:
df_produto_credito.display()

####Inserindo os dados na tabela bronze silver

In [0]:
(df_produto_credito.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dev_credit_risk.silver.dim_produto_credito")
)

####Consultando os dados com SQL no schema tabela delta silver

In [0]:
%sql
select * from dev_credit_risk.silver.dim_produto_credito